In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
import matplotlib.patches as mpatches

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

In [ ]:
norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"There are {n} frames")

In [ ]:
struct=["node", "filament", "wall", "void"]

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)


In [ ]:
def tracing(z_ref): #z_ref is the redshift for which the fraction should be 1

    trace = [[[],[],[],[]], [[],[],[],[]], [[],[],[],[]], [[],[],[],[]]] #nodes, filemnts, walls, voids

    index = zs.index(z_ref)
    snap_ref = snaps[index]

    data = il.snapshot.loadSubset(basePath, snap_ref, 'dm', fields = ['ParticleIDs', 'Coordinates'])
    ids_ref = data["ParticleIDs"]
    coords_ref = data['Coordinates']/1000

    print(f"At redshift z={z_ref}....")
    print(f"Total DM particles: {len(ids_ref)}")

    bool_filament = filaments[index].astype(bool)
    bool_wall = walls[index].astype(bool)
    bool_node = nodes[index].astype(bool)
    bool_void = voids[index].astype(bool)

    grid_x = np.floor(coords_ref[:, 0] / dl).astype(int) % res
    grid_y = np.floor(coords_ref[:, 1] / dl).astype(int) % res
    grid_z = np.floor(coords_ref[:, 2] / dl).astype(int) % res

    in_node_mask = bool_node[grid_x, grid_y, grid_z]
    node_particle_ids = ids_ref[in_node_mask]
    print(f"Particles in nodes: {len(node_particle_ids)}")

    in_filament_mask = bool_filament[grid_x, grid_y, grid_z]
    filament_particle_ids = ids_ref[in_filament_mask]
    print(f"Particles in filaments: {len(filament_particle_ids)}")

    in_wall_mask = bool_wall[grid_x, grid_y, grid_z]
    wall_particle_ids = ids_ref[in_wall_mask]
    print(f"Particles in walls: {len(wall_particle_ids)}")

    in_void_mask = bool_void[grid_x, grid_y, grid_z]
    void_particle_ids = ids_ref[in_void_mask]
    print(f"Particles in voids: {len(void_particle_ids)}")

    id_list = [node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]
    struct = ['Node', 'Filament', 'Wall', 'Void']

    h = 1
    j = 0
    for i in snaps:
        if j % 6 == 0 :
            print(f"Working on it {i}.... ({h}/3)")
            h+=1

        data = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs','Coordinates'])
        ids = data["ParticleIDs"]
        coord = data["Coordinates"]/1000

        sort_idx = np.argsort(ids)
        sorted_ids = ids[sort_idx]

        for g, k in enumerate(id_list):
            sample_size = np.size(k)
            
            print(f"Taking care of {struct[g]}s ....")

            match_indices = np.searchsorted(sorted_ids, k)
            original_indices = sort_idx[match_indices]
            matched_coords = coord[original_indices]

            grid_coord = (matched_coords//dl).astype(int)

            xx = grid_coord[:,0] % res
            yy = grid_coord[:,1] % res
            zz = grid_coord[:,2] % res

            is_node = nodes[j][xx, yy, zz] == 1
            is_filament = filaments[j][xx, yy, zz] == 1
            is_wall = walls[j][xx, yy, zz] == 1
            is_void = voids[j][xx, yy, zz] == 1

            trace[g][0].append(np.sum(is_node) / sample_size)
            trace[g][1].append(np.sum(is_filament) / sample_size)
            trace[g][2].append(np.sum(is_wall) / sample_size)
            trace[g][3].append(np.sum(is_void) / sample_size)

        j+=1

    print("Done!")
    
    return trace

In [ ]:
history = tracing(0.0)

In [ ]:
np.save('history.npy', history)

In [ ]:
fig = plt.figure(figsize=(12,8))

plt.style.use('seaborn-v0_8-colorblind')

for i in range(4):

    ax = fig.add_subplot(2,2,i+1)

    ax.plot(zs, history[i][0], label = "nodes", marker = "x")
    ax.plot(zs, history[i][1], label = "filaments", marker = "o")
    ax.plot(zs, history[i][2], label = "walls", marker = "*")
    ax.plot(zs, history[i][3], label = "voids", marker = "s")

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Origin of {struct[i]} particles")
    ax.legend()

plt.tight_layout();

plt.rcdefaults()

In [ ]:
evolution = tracing(10.0)

In [ ]:
np.save('evolution.npy', evolution)

In [ ]:
fig = plt.figure(figsize=(8,12))

plt.style.use('seaborn-v0_8-colorblind')

for i in range(1,4):

    ax = fig.add_subplot(3,1,i)

    ax.plot(zs, evolution[i][0], label = "nodes", marker = "x")
    ax.plot(zs, evolution[i][1], label = "filaments", marker = "o")
    ax.plot(zs, evolution[i][2], label = "walls", marker = "*")
    ax.plot(zs, evolution[i][3], label = "voids", marker = "s")

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Evolution of {struct[i]} particles")
    ax.legend()

plt.tight_layout();

plt.rcdefaults()